In [ ]:

import geopandas as gpd
import pandas as pd
import rasterio
from pathlib import Path
import xarray as xr
import rioxarray

In [ ]:
refpoints_csv = "..//..//WRIJ_RR_Unpaved_methode_01_data//rr_data_scenarios//scenarios//#SCENARIO#//#SCENARIO#_gebiedsindeling_RR_KNOPEN_tbv_Onderrand.csv"              # jouw CSV
raster_folder = "..//..//WRIJ_RR_Unpaved_methode_01_data//rr_data_scenarios//scenarios//#SCENARIO#//seepage//"      # map met ASC bestanden
output_csv = "..//..//WRIJ_RR_Unpaved_methode_01_data//rr_data_scenarios//scenarios//#SCENARIO#//seepage//kwel_per_RR_knoop.csv"

In [ ]:
crs = "EPSG:28992"  # waarschijnlijk RD New

In [ ]:
# selectie_gebied = 0 # Oude IJssel
# selectie_gebied = 1 # West
# selectie_gebied = 2 # Centraal
# selectie_gebied = 3 # Oost

selectie_gebieden = [1, 2]

scenarios = ["REF", "SCEN"]
# scenarios = ["REF"]

start_date = "2012-4-1"
end_date = "2018-12-1"

# path to the package containing the dummy-data
dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\"

In [ ]:
date_range_data = pd.date_range(start_date, end_date, freq="MS")

In [ ]:
for scenario in [scenarios[0]]:
    print(scenario)
    dir_seepage = Path(raster_folder.replace("#SCENARIO#", scenario))
    data_arrays = []
    for date in date_range_data:
        seepage_raster_filename = f"{scenario}_FLUX_L1L2_{date.strftime('%Y%m')}_MMD.ASC"
        da = rioxarray.open_rasterio(Path(dir_seepage, seepage_raster_filename), masked=True).squeeze("band")
        data_arrays.append(da)
    ds = xr.concat(data_arrays, dim="time")
    ds = ds.assign_coords(time=date_range_data)

In [ ]:
ds

In [ ]:
ds = xr.concat(data_arrays, dim="time")
ds = ds.assign_coords(time=date_range_data[:2])

In [ ]:
# 1. Lees puntenbestand

df = pd.read_csv(points_csv, sep=";")

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.xcoor, df.ycoor),
    crs=crs
)

In [ ]:
# 2. Alle rasterbestanden ophalen

raster_files = sorted(Path(raster_folder).glob("*.ASC"))

# lege lijst voor resultaten
results = []

In [ ]:
# 3. Loop over alle maanden
# ---------------------------
for raster_path in raster_files:
    print(f"Verwerken: {raster_path.name}")
    
    # maand uit bestandsnaam (pas dit aan!)
    month_name = raster_path.stem.split("_")[3]
    
    with rasterio.open(raster_path) as src:
        
        # check CRS
        if src.crs != gdf.crs:
            gdf = gdf.to_crs(src.crs)
        
        # sample rasterwaarden op puntlocaties
        coords = [(geom.x, geom.y) for geom in gdf.geometry]
        values = list(src.sample(coords))
        
        # rasterio geeft arrays terug → pak eerste waarde
        values = [val[0] if val is not None else None for val in values]
    
    # kopie maken voor deze maand
    temp = gdf.copy()
    temp["kwel"] = values
    temp["maand"] = month_name
    
    # ---------------------------
    # 4. Gemiddelde per subgebied
    # ---------------------------
    grouped = (
        temp
        .groupby("ID_RR_KNOOP")["kwel"]
        .mean()
        .reset_index()
    )
    
    grouped["maand"] = month_name
    
    results.append(grouped)

In [ ]:
final_df = pd.concat(results, ignore_index=True)

final_df.to_csv(output_csv, index=False)